Initialisation du chemin du projet

In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("Projet :", PROJECT_ROOT)
print("Src    :", SRC_PATH)

Projet : c:\Users\gnove\Documents\ml-housing-project
Src    : c:\Users\gnove\Documents\ml-housing-project\src


Chargement des donnÃ©es

In [5]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing = fetch_california_housing(as_frame=True)
df = housing.frame

X = df.drop(columns=["MedHouseVal"])
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


Entrainement du model

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def build_random_forest_pipeline(n_estimators):
    return Pipeline(
        [
            (
                "preprocessing",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
            ),
            (
                "model",
                RandomForestRegressor(
                    n_estimators=n_estimators,
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ]
    )


model_v1 = build_random_forest_pipeline(n_estimators=20)
model_v1.fit(X_train, y_train)

Evaluation du model

In [7]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def evaluate(model, X_test, y_test):
    predictions = model.predict(X_test)
    return {
        "mae": float(mean_absolute_error(y_test, predictions)),
        "rmse": float(np.sqrt(mean_squared_error(y_test, predictions))),
        "r2": float(r2_score(y_test, predictions)),
    }


metrics_v1 = evaluate(model_v1, X_test, y_test)
metrics_v1

{'mae': 0.3373539152131783,
 'rmse': 0.5149931062641127,
 'r2': 0.7976067747524215}

Sauvegarde du modÃ¨le V1 et ses mÃ©triques

In [10]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
print(PROJECT_ROOT)

c:\Users\gnove\Documents\ml-housing-project


In [11]:
import json

import joblib

models_dir = PROJECT_ROOT / "artifacts" / "models"
metrics_dir = PROJECT_ROOT / "artifacts" / "metrics"

models_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(model_v1, models_dir / "model_v1.joblib")

with open(metrics_dir / "metrics_v1.json", "w", encoding="utf-8") as file:
    json.dump(metrics_v1, file, indent=2)

print("Modele V1 sauvegarde")

Modele V1 sauvegarde


Entrainer une version V2

In [ ]:
model_v2 = build_random_forest_pipeline(n_estimators=50)
model_v2.fit(X_train, y_train)

metrics_v2 = evaluate(model_v2, X_test, y_test)
metrics_v2

Sauvegarder la version V2

In [15]:
joblib.dump(model_v2, models_dir / "model_v2.joblib")

with open(metrics_dir / "metrics_v2.json", "w", encoding="utf-8") as file:
    json.dump(metrics_v2, file, indent=2)

print("ModÃ¨le V2 sauvegarde")

ModÃ¨le V2 sauvegarde


Choix du modÃ¨le actif

In [16]:
import shutil

# Exemple : on choisit V2 comme modÃ¨le actif
shutil.copy(
    models_dir / "model_v2.joblib",
    models_dir / "model_latest.joblib",
)

with open(metrics_dir / "metrics_v2.json", "r", encoding="utf-8") as src:
    latest_metrics = json.load(src)

with open(metrics_dir / "metrics_latest.json", "w", encoding="utf-8") as dst:
    json.dump(latest_metrics, dst, indent=2)

print("ModÃ¨le actif : model_latest.joblib")

ModÃ¨le actif : model_latest.joblib


VÃ©rification des fichiers gÃ©nÃ©rÃ©s

In [17]:
models_dir = PROJECT_ROOT / "artifacts" / "models"
metrics_dir = PROJECT_ROOT / "artifacts" / "metrics"

list(models_dir.iterdir()), list(metrics_dir.iterdir())

([WindowsPath('c:/Users/gnove/Documents/ml-housing-project/artifacts/models/model_latest.joblib'),
  WindowsPath('c:/Users/gnove/Documents/ml-housing-project/artifacts/models/model_v1.joblib'),
  WindowsPath('c:/Users/gnove/Documents/ml-housing-project/artifacts/models/model_v2.joblib')],
 [WindowsPath('c:/Users/gnove/Documents/ml-housing-project/artifacts/metrics/metrics_latest.json'),
  WindowsPath('c:/Users/gnove/Documents/ml-housing-project/artifacts/metrics/metrics_v1.json'),
  WindowsPath('c:/Users/gnove/Documents/ml-housing-project/artifacts/metrics/metrics_v2.json')])